# Downsample factor vs. mosaic quality

Visual comparison: the same real 15-FOV block
(`notebooks/tests/tissue_thickness/01_elevation_heatmap.ipynb`'s own
section 8 block, straddling a real tissue edge), FFC-corrected (same
min-projection field that notebook established as best), one single z-plane
(z=25 um), stitched at `DOWNSAMPLE_FACTOR` = 1, 2, 4, 8, 16, 32 -- so you
can judge by eye which resolution still looks acceptable before it's used
for anything (the elevation-heatmap pipeline, a production mosaic, etc.).

**Self-contained** (`NOTEBOOK_GUIDELINES.md` #7): the small FFC field file
(~18 MB) is copied in from `01_elevation_heatmap.ipynb`'s own cache the
first time it's needed (provenance noted where it's loaded below) rather
than read live from that notebook's cache path on every run -- the block
definition (FOV ids, grid position) is copied as a literal constant for
the same reason, not re-derived by depending on that notebook's own
in-memory state.

**Only one frame per FOV needs reading this time** (not a full z-stack, and
not the full 882-FOV interior population) -- 15 real raw reads, cheap, no
SLURM needed. Each is FFC-corrected once at full raw resolution, then
downsampled by successive 2x block-averaging (1 -> 2 -> 4 -> 8 -> 16 -> 32)
-- mean-of-equal-size-groups is associative, so this is mathematically
identical to downsampling directly from the corrected raw frame by each
factor, just cheaper (one correction, one chain of halvings, instead of
6 independent recomputations).

## 1 -- Setup

In [ ]:
%matplotlib inline
# %matplotlib widget  # uncomment for interactive pan/zoom (ipympl)

import os
import sys
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.measure import block_reduce
from PIL import Image

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/),
# same convention as notebooks/before_imaging/regular/ (3 levels).
MERCI_DIR = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config import ExperimentConfig
from MERci.common.metadata import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io import read_image_frames
from MERci.acquisition.configs import get_fov_geometry, find_frame_table_for_hal_config
from MERci.acquisition.configs import load_microscope_orientation, apply_microscope_orientation
from MERci.analysis.ffc import compute_mosaic_crop_px, apply_ffc, load_ffc_field
from MERci.visualization import get_merci_figures_dir

NOTEBOOK_NAME = "downsample_mosaic"

PLOT_TITLE_FONTSIZE, PLOT_LABEL_FONTSIZE = 13, 11
PLOT_TICK_FONTSIZE, PLOT_LEGEND_FONTSIZE = 10, 10

## 2 -- Parameters

In [ ]:
# Same real dataset as notebooks/tests/tissue_thickness/01_elevation_heatmap.ipynb.
SAMPLE_DIR = Path("/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/lineage_tracing/experiments/LT066_sample_01/merfish")

MICROSCOPE = "ST2"
OBJECTIVE  = "60X"
CHANNEL_NM = 405.0   # DAPI
Z_UM       = 25.0    # single z-plane to compare across downsample factors

DOWNSAMPLE_FACTORS = [1, 2, 4, 8, 16, 32]

# The exact same section-8 block from 01_elevation_heatmap.ipynb (15 real
# FOVs straddling a real tissue edge) -- copied as a literal constant
# rather than re-derived, so this notebook doesn't depend on that one
# having been run first.
BLOCK_FOV_IDS   = [9, 39, 40, 41, 42, 43, 78, 79, 80, 81, 82, 83, 84, 85, 86]
BLOCK_R0, BLOCK_C0 = 0, 1
BLOCK_GRID_ROWS = BLOCK_GRID_COLS = 5

CACHE_DIR = SAMPLE_DIR / "analysis" / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "tests", NOTEBOOK_NAME, subfolder="downsample_mosaic")

SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(SAMPLE_DIR / "MERci")
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)

pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE, OBJECTIVE)

config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt=SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix=".zarr", microscope=MICROSCOPE,
    pixel_size_um=pixel_size_um, image_size_px=image_size_px,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                image_suffix=config.image_suffix)

MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MICROSCOPE_ORIENTATION = load_microscope_orientation(MICROSCOPE, MICROSCOPE_ORIENTATION_DIR)

cells_round_id = meta.round_for_imaging_type("cells")
round_info = meta.rounds[cells_round_id]
positions = {fov_id: meta.fovs[fov_id].position
             for fov_id in round_info.fov_files if round_info.fov_files[fov_id]}

xy = np.array([positions[i] for i in positions])
x0, y0 = xy[:, 0].min(), xy[:, 1].min()
grid_indices = {}
for fov_id, (x, y) in positions.items():
    col = int(round((x - x0) / config.step_size_um))
    row = int(round((y - y0) / config.step_size_um))
    grid_indices[fov_id] = (row, col)

for s in meta.series_for_round(cells_round_id):
    if s.hal_config:
        ft_path = find_frame_table_for_hal_config(config.settings_dir / s.hal_config, config.metadata_dir)
        break
frame_table = pd.read_csv(ft_path, index_col=0)
channel_frames = frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)].sort_values("z")
z_match = channel_frames[np.isclose(channel_frames["z"], Z_UM)]
if z_match.empty:
    raise ValueError(f"No exact frame at z={Z_UM} um in this round's {CHANNEL_NM} nm z-grid")
z_frame_idx = int(z_match.index[0])

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"Block        : {len(BLOCK_FOV_IDS)} FOV(s), rows [{BLOCK_R0},{BLOCK_R0+BLOCK_GRID_ROWS}) cols [{BLOCK_C0},{BLOCK_C0+BLOCK_GRID_COLS})")
print(f"z = {Z_UM} um -> frame_idx = {z_frame_idx}")
print(f"Cache  : {CACHE_DIR}")
print(f"Figures: {FIGURES_DIR}")

## 3 -- FFC field (copied in from `01_elevation_heatmap.ipynb`'s own cache)

Same min-projection field that notebook's own comparison
(`notebooks/tests/calculate_ffc/01_compare_ffc_methods.ipynb`) established
as best -- copied in once (NOTEBOOK_GUIDELINES.md #7: small file, copy
rather than depend on the live path) so this notebook stays runnable on
its own.

In [ ]:
ffc_field_path = CACHE_DIR / f"ffc_field_v2_interior_min_{int(CHANNEL_NM)}nm.npz"
if not ffc_field_path.exists():
    source_path = SAMPLE_DIR / "analysis" / "cache" / "elevation_heatmap" / f"ffc_field_v2_interior_min_{int(CHANNEL_NM)}nm.npz"
    if not source_path.exists():
        raise FileNotFoundError(
            f"{source_path} not found -- run 01_elevation_heatmap.ipynb's section 6 first "
            f"(builds and caches this FFC field), or point ffc_field_path at another cached field."
        )
    shutil.copy2(source_path, ffc_field_path)
    print(f"Copied FFC field from {source_path} -> {ffc_field_path}")

ffc_field, ffc_meta = load_ffc_field(ffc_field_path)
print(f"Loaded FFC field: {ffc_field_path}  ({ffc_meta})")

## 4 -- Read + FFC-correct each block FOV's z=25 frame (raw resolution)

One raw read per FOV (15 total) -- cheap. Corrected once at full
resolution; every downsample level below is derived from this same
corrected frame by successive 2x block-averaging, not by re-reading or
re-correcting.

In [ ]:
corrected_frames = {}
for fov_id in BLOCK_FOV_IDS:
    fpath = round_info.fov_files[fov_id][0]
    raw = read_image_frames(fpath, [z_frame_idx])[0]
    oriented = apply_microscope_orientation(raw, **MICROSCOPE_ORIENTATION)
    corrected_frames[fov_id] = apply_ffc(oriented, ffc_field)

print(f"{len(corrected_frames)} FOV(s) read + FFC-corrected at full "
      f"{corrected_frames[BLOCK_FOV_IDS[0]].shape} resolution.")

## 5 -- Build + display the mosaic at each downsample factor

In [ ]:
def center_crop(arr, crop_px):
    if crop_px == 0:
        return arr
    return arr[crop_px:-crop_px, crop_px:-crop_px]

def stitch_by_grid(tiles, r0, c0, n_rows, n_cols, crop_px, fill=0):
    cropped = {f: center_crop(t, crop_px) for f, t in tiles.items()}
    tile_h, tile_w = next(iter(cropped.values())).shape
    canvas = np.full((n_rows * tile_h, n_cols * tile_w), fill, dtype=np.float32)
    for fov_id, tile in cropped.items():
        r, c = grid_indices[fov_id]
        rr, cc = r - r0, c - c0
        canvas[rr*tile_h:(rr+1)*tile_h, cc*tile_w:(cc+1)*tile_w] = tile
    return canvas

def to_uint8(arr, vmin, vmax):
    scaled = (arr.astype(np.float64) - vmin) / max(vmax - vmin, 1e-9) * 255
    return np.clip(scaled, 0, 255).astype(np.uint8)

crop_px_raw = compute_mosaic_crop_px(config)
print(f"crop_px_raw = {crop_px_raw}")

# Shared display scale across every factor (pooled from the full-res
# corrected frames) so brightness differences reflect real signal, not a
# per-factor auto-contrast that would hide true detail loss.
pooled = np.concatenate([f.ravel() for f in corrected_frames.values()])
vmin_d, vmax_d = np.percentile(pooled, [1.0, 99.0])
print(f"Shared display scale (p1-p99): [{vmin_d:.0f}, {vmax_d:.0f}]")

# Every mosaic below is displayed at this same physical size regardless of
# its native resolution (see the Notes section at the end for why) -- the
# native, full-resolution array is saved straight to PNG via PIL (not
# matplotlib's heavier imsave path) and then dropped, only the small
# display-sized version is kept in memory for the comparison plot below.
DISPLAY_SIZE_PX = 700

current = {f: img.astype(np.float32) for f, img in corrected_frames.items()}
current_factor = 1
mosaics = {}
canvas_shapes = {}
for factor in DOWNSAMPLE_FACTORS:
    step = factor // current_factor
    if step > 1:
        current = {f: block_reduce(img, (step, step), func=np.mean) for f, img in current.items()}
        current_factor = factor
    crop_px = crop_px_raw // factor
    tiles = {f: to_uint8(img, vmin_d, vmax_d) for f, img in current.items()}
    canvas = stitch_by_grid(tiles, BLOCK_R0, BLOCK_C0, BLOCK_GRID_ROWS, BLOCK_GRID_COLS, crop_px, fill=0).astype(np.uint8)
    canvas_shapes[factor] = canvas.shape

    png_path = FIGURES_DIR / f"{NOTEBOOK_NAME}.factor{factor:02d}.png"
    Image.fromarray(canvas).save(png_path)

    display_img = Image.fromarray(canvas).resize((DISPLAY_SIZE_PX, DISPLAY_SIZE_PX), resample=Image.BILINEAR)
    mosaics[factor] = np.asarray(display_img)
    del canvas, tiles

    print(f"factor={factor:2d}: tile={current[BLOCK_FOV_IDS[0]].shape[0]:5d}px  "
          f"canvas={canvas_shapes[factor][0]}x{canvas_shapes[factor][1]}  saved -> {png_path}")

In [ ]:
for factor in DOWNSAMPLE_FACTORS:
    fig, ax = plt.subplots(figsize=(7, 7), dpi=100)
    ax.imshow(mosaics[factor], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"downsample = {factor}  (native canvas {canvas_shapes[factor][0]}x{canvas_shapes[factor][1]} px)",
                 fontsize=PLOT_TITLE_FONTSIZE)
    ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    plt.show()

## Notes

Each mosaic above is rendered at the **same physical display size**
(figsize/dpi held constant) regardless of the underlying array
resolution -- that's the fair comparison for "does this still look good",
since a downsample=1 mosaic isn't actually viewed at its native 10000+ px
size in practice either. The full-resolution PNG for each factor is also
saved to `{FIGURES_DIR}` (`{NOTEBOOK_NAME}.factor<N>.png`) if you want to
zoom into the real pixels directly outside the notebook.